# Charts with seaborn

Python stage, lesson P22. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** see that seaborn draws on a matplotlib Axes and that its bar chart summarizes rows; draw a grouped comparison with `hue`; draw a histogram; apply a theme and check that the numbers did not change.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

def check_chart(label, name, bars=None, line=None, zero=False, title=None, xlabel=False, ylabel=False, no_bars=False, traps=None):
    """Look inside the Axes called `name`: bar heights, line values, the bottom of the y axis, the title and the labels."""
    ax = globals().get(name)
    if ax is None or not hasattr(ax, "get_title"):
        RESULTS[label] = None
        print(f"{label}: give {name} the Axes of your chart, then run this cell again.")
        return False
    problems = []
    try:
        heights = [round(float(p.get_height()), 2) for p in ax.patches if p.get_height() != 0]  # legend swatches are zero-height patches
        values = [round(float(y), 2) for y in ax.lines[0].get_ydata()] if ax.lines else []
        if bars is not None and not same(heights, [round(float(b), 2) for b in bars]):
            problems.append(f"the bars should have the heights {bars!r}, but they are {heights!r}")
        if no_bars and heights:
            problems.append("a line chart has no bars; this one has " + str(len(heights)))
        if line is not None and not same(values, [round(float(v), 2) for v in line]):
            problems.append(f"the line should pass through {line!r}, but it passes through {values!r}")
        if zero and ax.get_ylim()[0] > 0:
            problems.append(f"the y axis starts at {ax.get_ylim()[0]:g}; bars must start at 0")
        if title is True and not ax.get_title().strip():
            problems.append("the chart has no title")
        if isinstance(title, str) and title.lower() not in ax.get_title().lower():
            problems.append(f"the title should name {title!r}; it is {ax.get_title()!r}")
        if xlabel and not ax.get_xlabel().strip():
            problems.append("the x axis has no label")
        if ylabel and not ax.get_ylabel().strip():
            problems.append("the y axis has no label")
    except Exception as error:
        problems.append(f"the Axes could not be read ({type(error).__name__})")
    if not problems:
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. " + "; ".join(problems) + ".")
    return False

# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, MAGENTA, GOLD, GREEN = "#0072b2", "#d6479a", "#ad9100", "#00a07a"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, MAGENTA, GOLD, GREEN]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# One row for each order line with its amount, store, month and category, as built in lessons P13, P15 and P17.
price_of = dict(zip(products["product_id"], products["unit_price_usd"]))
lines = order_items.assign(price=order_items["product_id"].map(price_of))
lines = lines.assign(amount=lines["quantity"] * lines["price"] - lines["discount_usd"])
sales = lines.merge(orders[["order_id", "store_id", "ordered_at"]], on="order_id", validate="many_to_one")
sales = sales.merge(products[["product_id", "category"]], on="product_id", validate="many_to_one")
sales = sales.assign(month=pd.to_datetime(sales["ordered_at"]).dt.month)
# Three tables for the charts: the sales of each store and month, of each category, and of each order.
store_names = dict(zip(stores["store_id"], stores["store_name"]))
store_month = sales.groupby(["store_id", "month"])["amount"].sum().reset_index()
store_month = store_month.assign(store=store_month["store_id"].map(store_names))
by_category = sales.groupby("category")["amount"].sum().reset_index()
order_totals = sales.groupby("order_id")["amount"].sum()

## Unit 1: seaborn on an Axes, and what barplot does with rows

**Predict first.** what type will `sns.barplot` return, is it the same object as `ax`, and what will the four bar heights be?

In [ ]:
fig, ax = plt.subplots()
back = sns.barplot(data=by_category, x="category", y="amount", ax=ax)
print(type(back).__name__, back is ax)
print([round(float(p.get_height()), 2) for p in ax.patches])

**Predict first.** `sales` has 3,287 rows, many for each category. What will the four bar heights be, and how many lines will the Axes hold besides the bars?

In [ ]:
fig, ax = plt.subplots()
sns.barplot(data=sales, x="category", y="amount", seed=1, ax=ax)  # seed: the same random draw for the error bars every time
print([round(float(p.get_height()), 2) for p in ax.patches], len(ax.lines))

## Unit 2: hue: one color for each value of a column

**Predict first.** there are 3 stores and 6 months. How many bars with a height above 0 will the chart have, and what will the last three be?

In [ ]:
fig, ax = plt.subplots()
sns.barplot(data=store_month, x="month", y="amount", hue="store", ax=ax)
ax.legend(ncols=3, loc="lower center", bbox_to_anchor=(0.5, 1))  # the legend above the bars, so it hides none
bars = [round(float(p.get_height()), 1) for p in ax.patches if p.get_height() > 0]
print(len(bars))
print(bars[-3:])

## Unit 3: histplot: the shape of one number

**Predict first.** how many bars will there be, what will their heights add up to, and what will the last bar hold?

In [ ]:
fig, ax = plt.subplots()
sns.histplot(order_totals, bins=10, ax=ax)
counts = [int(p.get_height()) for p in ax.patches]
print(len(counts), sum(counts))
print(counts)

## Unit 4: set_theme: a new look, the same numbers

**Predict first.** will the four heights be the same as in unit 1, and what will the title say?

In [ ]:
sns.set_theme(style="whitegrid", palette=[BLUE, MAGENTA, GOLD, GREEN], font_scale=0.9,
              rc={"figure.figsize": (4.5, 2.8), "figure.constrained_layout.use": True})
fig, ax = plt.subplots()
sns.barplot(data=by_category, x="category", y="amount", ax=ax)
share = by_category.loc[1, "amount"] / by_category["amount"].sum()
ax.set_title(f"Coffee made {share:.0%} of the sales")
print([round(float(p.get_height()), 2) for p in ax.patches])
print(ax.get_title())

## Unit 5: Check: seaborn and pandas agree

**Predict first.** do the bars of the seaborn chart equal the totals that pandas computes from the order lines?

In [ ]:
pandas_totals = sales.groupby("category")["amount"].sum().tolist()
print(pandas_totals == [round(float(p.get_height()), 2) for p in ax.patches])

## Finish it

### Exercise 1: Finish it

**Task:** How do the three cafes compare in each payment method? `pay` holds the orders of each payment method and store, one row for each pair, with the store names. The call below draws one bar for each method. Change it so that each method has one bar for each store, then add a title, a label on the x axis and a label on the y axis. Expected: the chart in `ax_1`.

**Expected:** the Axes `ax_1`: nine bars, three for each payment method, a title and both axis labels.

In [ ]:
pay = orders.groupby(["payment_method", "store_id"]).size().reset_index(name="orders")
pay = pay.assign(store=pay["store_id"].map(store_names))  # given: one row for each payment method and store
fig, ax_1 = plt.subplots()
sns.barplot(data=pay, x="payment_method", y="orders", ax=ax_1)  # given: one bar for each method
# change the call above, then add the title, the x label and the y label here, on ax_1

In [ ]:
_ = check_chart("Exercise 1", "ax_1", bars=[421, 91, 217, 400, 107, 209, 180, 55, 120], title=True, xlabel=True, ylabel=True)

<details>
<summary>Hint</summary>

The column that splits the bars is `store`, and unit 2 shows the argument that takes it.

</details>

<details>
<summary>Solution</summary>

```python
pay = orders.groupby(["payment_method", "store_id"]).size().reset_index(name="orders")
pay = pay.assign(store=pay["store_id"].map(store_names))
fig, ax_1 = plt.subplots()
sns.barplot(data=pay, x="payment_method", y="orders", hue="store", ax=ax_1)
ax_1.set_title("Card is the most used payment method in every store")
ax_1.set_xlabel("Payment method")
ax_1.set_ylabel("Orders, January to June")
```

The nine bars are card 421, 400 and 180, cash 91, 107 and 55, mobile 217, 209 and 120, for Old Town, Campus and Harbor. Without `hue`, seaborn averages the three stores of each method into one bar of 333.7, 84.3 and 182.0, and the check cell flags it.

</details>

## On your own

### Exercise 2: On your own

**Task:** How are the orders of Harbor, store 3, spread by value? Before you start, say in one sentence what the chart shows: one number, the value of an order, for Harbor only. Draw a histogram of those order values with 6 bins on an Axes called `ax_2`, with a title and a label on each axis. Write the code in the empty cell of the notebook. Expected: six bars. Write the code in the empty cell below and set `answer_2`.

**Expected:** the Axes `ax_2`: six bars that count Harbor's 355 orders, a title and both axis labels.

In [ ]:
_ = check_chart("Exercise 2", "ax_2", bars=[118, 118, 64, 36, 12, 7], title=True, xlabel=True, ylabel=True)

<details>
<summary>Hint</summary>

`sales` has the `store_id` of each line. Add up the amounts of each order for store 3, as `order_totals` does for all orders.

</details>

<details>
<summary>Solution</summary>

```python
harbor_totals = sales[sales["store_id"] == 3].groupby("order_id")["amount"].sum()
fig, ax_2 = plt.subplots()
sns.histplot(harbor_totals, bins=6, ax=ax_2)
ax_2.set_title("Most Harbor orders are worth less than 15 dollars")
ax_2.set_xlabel("Order value (USD)")
ax_2.set_ylabel("Orders")
```

The six bars hold 118, 118, 64, 36, 12 and 7 orders, 355 in all, for values from 3.00 to 38.75. Drawing `order_totals`, with the orders of every store, gives 1,780, 18, 1, 0, 0 and 1, because order 2075 stretches the axis, and the check cell says so.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague's chart is titled `Coffee sells the most`, but a reader has to hunt for coffee's bar among the others. Fix the lines so that the bars run from the largest to the smallest sales. Expected: the chart in `ax_3`, the four bars in that order.

**Expected:** the Axes `ax_3`: four bars from the largest to the smallest, a title and both axis labels.

In [ ]:
ranking = by_category  # the colleague's table
fig, ax_3 = plt.subplots()
sns.barplot(data=ranking, x="category", y="amount", ax=ax_3)
ax_3.set_title("Coffee sells the most")
ax_3.set_xlabel("Category")
ax_3.set_ylabel("Sales, January to June (USD)")

In [ ]:
_ = check_chart("Exercise 3", "ax_3", bars=[10807.0, 4890.5, 3824.25, 3232.5], title=True, xlabel=True, ylabel=True)

<details>
<summary>Hint</summary>

Print `by_category`. In which order are the rows, and in which order does seaborn draw the bars? Lesson P14 shows how to put rows in the order of a column.

</details>

<details>
<summary>Solution</summary>

```python
ranking = by_category.sort_values("amount", ascending=False)
fig, ax_3 = plt.subplots()
sns.barplot(data=ranking, x="category", y="amount", ax=ax_3)
ax_3.set_title("Coffee sells the most")
ax_3.set_xlabel("Category")
ax_3.set_ylabel("Sales, January to June (USD)")
```

The bars are 10,807.00, 4,890.50, 3,824.25 and 3,232.50: coffee, food, bakery, tea. The colleague's table was in alphabetical order, and seaborn draws the categories in the order of the table, so the chart did not match its title. Sorting the table first, as in lesson P14, fixes it.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")